# Colab Setup

**For Colab users:** Uncomment and run the cell below to mount Drive.  
**For local users:** Skip this cell.

In [ ]:
# Uncomment for Colab:
#from google.colab import drive
#drive.mount('/content/drive')

## Setup: Dependency Verification and Installation

This cell checks for the presence of essential Python libraries (like `numpy`, `torch`, `musdb`, `nbformat`, etc.).
If any required library is not found, it attempts to install it automatically using `pip`.
It also verifies the availability of PyTorch with CUDA, which is crucial for GPU-accelerated training.

In [ ]:
# --- 1. Verify and install dependencies ---
print("Verifying and installing missing packages if necessary...")

packages_to_check = [
    'numpy', 'matplotlib', 'librosa', 'tqdm', 'sklearn', 'stempeg', 'torch', 'torchvision', 'torchaudio', 'musdb'
]

for package in packages_to_check:
    try:
        __import__(package)
        print(f"  ✅ {package} is installed.")
    except ImportError:
        print(f"  ❌ {package} is NOT installed. Attempting to install...")
        try:
            import sys
            import subprocess
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])
            __import__(package)
            print(f"  ✅ {package} is now installed.")
        except Exception as e:
            print(f"  ❌ Failed to install {package}: {e}")

# Special check for PyTorch CUDA
print("\n--- PyTorch CUDA status ---")
try:
    import torch
    if torch.cuda.is_available():
        print(f"  ✅ PyTorch with CUDA (version {torch.version.cuda}) is available.")
        print(f"     CUDA Device Name: {torch.cuda.get_device_name(0)}")
    else:
        print("  ⚠️ PyTorch is installed, but CUDA is NOT available.")
except ImportError:
    print("  ❌ PyTorch is not installed.")

print("Verification complete.")

## Imports and Environment Setup

- Import required libraries (torch, numpy, matplotlib, etc.)

- Set device (CPU/GPU)

In [ ]:
import sys
from pathlib import Path
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from IPython.display import Audio, display
import importlib

# Detect environment and set Project Root
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Final_Project_Deep_Learning')
    print(f"✅ Colab Project Root: {PROJECT_ROOT}")
else:
    # Local: use current working directory
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / 'modelA.ipynb').exists():
        for p in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
            if (p / 'modelA.ipynb').exists():
                PROJECT_ROOT = p
                break

os.chdir(PROJECT_ROOT)

# Define data and checkpoint directories
DATA_DIR = PROJECT_ROOT / "data"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Add to sys.path for imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Import project modules
import models.utils as utils
importlib.reload(utils)
from models import model_A as ma

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

# Device setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"\nConfiguration Complete:")
print(f"   - Device: {device}")
print(f"   - Working Directory: {os.getcwd()}")
print(f"   - Data Directory: {DATA_DIR}")
print(f"   - Checkpoints Directory: {CHECKPOINT_DIR}")

## MUSDB18 Setup

**Quick Start:**
1. Download MUSDB18
2. Extract to project folder as `musdb18/`
3. Run preprocessing below

**Expected structure:**
```
musdb18/
  train/     (~100 songs)
  valid/     (~14 songs)
  test/      (~50 songs)
```

In [ ]:
# ============================================================================
# MUSDB18 PATH CONFIGURATION
# ============================================================================
# Auto-detect musdb18 folder in project directory
MUSDB18_PATH = PROJECT_ROOT / "musdb18"

if not MUSDB18_PATH.exists():
    print("⚠️  MUSDB18 folder not found!")
    print(f"   Expected location: {MUSDB18_PATH}")
    print("")
    print("📥 Download MUSDB18:")
    print("   1. Register at https://zenodo.org/record/1438122")
    print("   2. Download MUSDB18-HQ.zip (~22GB)")
    print("   3. Extract to project folder as 'musdb18'")
    print("")
    MUSDB18_PATH = None
else:
    # Check for required subfolders
    train_dir = MUSDB18_PATH / "train"
    valid_dir = MUSDB18_PATH / "valid"
    test_dir = MUSDB18_PATH / "test"
    
    if train_dir.exists() and test_dir.exists():
        num_train = len(list(train_dir.iterdir()))
        num_valid = len(list(valid_dir.iterdir())) if valid_dir.exists() else 0
        num_test = len(list(test_dir.iterdir()))
        print(f"✅ MUSDB18 dataset found: {MUSDB18_PATH}")
        print(f"   Train: {num_train} tracks")
        print(f"   Valid: {num_valid} tracks")
        print(f"   Test: {num_test} tracks")
    else:
        print(f"⚠️  Found musdb18 folder but missing train/test subfolders")
        print(f"   Path: {MUSDB18_PATH}")
        MUSDB18_PATH = None

# Initialize file lists (will be populated after preprocessing)
mix_files_stage1 = mix_files_stage2 = tgt_files_stage1 = tgt_files_stage2 = []

## Data Preprocessing

Create 2 chunk size versions (8s, 10s). Stage 1: vocals+other, Stage 2: all sources.

In [ ]:
# ============================================================================
# DATA PREPROCESSING: MUSDB18 → CHUNKED TRAINING DATA
# ============================================================================

PROCESS_DATA = True  # Set to False to skip preprocessing (use existing data)
SAMPLE_RATE = 22050  # Target sample rate for all audio

# Chunk configurations
CHUNK_CONFIGS = [
    {'duration': 8.0, 'name': 'chunks_8s'},   # Currently used for training
    {'duration': 10.0, 'name': 'chunks_10s'}, # For larger models
]

if PROCESS_DATA and MUSDB18_PATH:
    print(f"\n{'='*70}")
    print(f"PREPROCESSING {len(CHUNK_CONFIGS)} CHUNK CONFIGURATIONS")
    print(f"{'='*70}\n")
    
    for idx, config in enumerate(CHUNK_CONFIGS, 1):
        chunk_dir = DATA_DIR / config['name']
        
        # Skip if already exists
        if (chunk_dir / "stage1" / "train" / "mixture").exists():
            print(f"[{idx}/{len(CHUNK_CONFIGS)}] ⏭️  {config['name']} already exists, skipping...")
            continue
        
        print(f"\n[{idx}/{len(CHUNK_CONFIGS)}] 🔄 Processing {config['duration']}s chunks...")
        print(f"Output: {chunk_dir}")
        
        # Run preprocessing
        preprocessing_stats = utils.preprocess_musdb18(
            musdb18_path=MUSDB18_PATH,
            output_dir=chunk_dir,
            chunk_duration=config['duration'],
            sample_rate=SAMPLE_RATE,
            stage1_ratio=0.7,
            train_ratio=0.7,
            val_ratio=0.15,
            test_ratio=0.15
        )
        
        print(f"✅ {config['name']} complete!")
    
    print(f"\n{'='*70}")
    print("🎯 ALL PREPROCESSING COMPLETE!")
    print(f"{'='*70}")
    print("\nData structure:")
    print("  data/")
    for config in CHUNK_CONFIGS:
        print(f"    {config['name']}/")
        print(f"      stage1/ (train/val/test → mixture/target)")
        print(f"      stage2/ (train/val/test → mixture/target)")
    
    # Verify one chunk size as example
    print(f"\n{'='*70}")
    print("VERIFICATION (chunks_8s example)")
    print(f"{'='*70}\n")
    
    sample_dir = DATA_DIR / "chunks_8s"
    if sample_dir.exists():
        for stage in ['stage1', 'stage2']:
            print(f"{stage.upper()}:")
            for split in ['train', 'val', 'test']:
                mix_dir = sample_dir / stage / split / "mixture"
                if mix_dir.exists():
                    n = len(list(mix_dir.glob("*.npy")))
                    print(f"  {split:5s}: {n:,} chunks")
            print()

elif PROCESS_DATA and not MUSDB18_PATH:
    print("⚠️  Cannot process data: MUSDB18_PATH not set")
    print("   Please extract the dataset to the 'musdb18' folder")
    
else:
    print("ℹ️  Data preprocessing skipped (PROCESS_DATA = False)")
    print("   Using existing preprocessed data...")

## Model A: Two Architectures for Comparison

**Model A (LSTM)** 1️⃣: Sequential bidirectional LSTM with masking output.

**Model A (U-Net)** 2️⃣: 2D CNN encoder-decoder with skip connections.

In [ ]:
print("="*70)
print("MODEL A ARCHITECTURES")
print("="*70)

# Quick architecture preview
lstm_preview, _, _, _ = utils.initialize_model_a_lstm(device)
unet_preview, _, _, _ = utils.initialize_model_a_unet(device)

lstm_params = sum(p.numel() for p in lstm_preview.parameters())
unet_params = sum(p.numel() for p in unet_preview.parameters())

print("\n1️⃣ Model A (LSTM):")
print(f"   Parameters: {lstm_params:,}")
print(f"   Type: Bidirectional LSTM with masking")

print("\n2️⃣ Model A (U-Net):")
print(f"   Parameters: {unet_params:,}")
print(f"   Type: 2D CNN encoder-decoder")

# Clean up preview models
del lstm_preview, unet_preview
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n" + "="*70)

## Train Both Models - Stage 1

Sequential training: LSTM first, then U-Net

In [ ]:
# ============================================================================
# TRAIN BOTH MODEL A ARCHITECTURES - STAGE 1 (2 -> 1)
# ============================================================================

SKIP_TRAINING_STAGE1 = False  # Set to True to skip Stage 1 training
CHUNK_DURATION = 8.0          # Using 8-second chunks
TRAINING_DATA_DIR = DATA_DIR / f"chunks_{CHUNK_DURATION:.0f}s"

# Separate configs per model
TRAIN_CONFIG_LSTM = utils.get_training_config_lstm()
TRAIN_CONFIG_UNET = utils.get_training_config_unet()

print(f"\n{'='*70}")
print("STAGE 1 TRAINING: 2 → 1")
print(f"{'='*70}\n")

# Stage 1 checkpoints
ckpt_lstm_s1 = CHECKPOINT_DIR / f"model_a_lstm_stage1_{CHUNK_DURATION:.0f}s.pth"
ckpt_unet_s1 = CHECKPOINT_DIR / f"model_a_unet_stage1_{CHUNK_DURATION:.0f}s.pth"

print("1️⃣ Model A (LSTM) - Stage 1")
print("-" * 70)
model_lstm, processor_lstm, optimizer_lstm, loss_fn_lstm = utils.initialize_model_a_lstm(device)
hist_lstm_s1 = utils.train_model_stage(
    model=model_lstm,
    processor=processor_lstm,
    optimizer=optimizer_lstm,
    loss_fn=loss_fn_lstm,
    training_data_dir=TRAINING_DATA_DIR,
    stage="stage1",
    ckpt_path=ckpt_lstm_s1,
    device=device,
    train_config=TRAIN_CONFIG_LSTM,
    skip_training=SKIP_TRAINING_STAGE1
)

# Free LSTM GPU memory before training U-Net
import gc
model_lstm = model_lstm.to('cpu')
del optimizer_lstm
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

print("\n2️⃣ Model A (U-Net) - Stage 1")
print("-" * 70)
model_unet, processor_unet, optimizer_unet, loss_fn_unet = utils.initialize_model_a_unet(device)
hist_unet_s1 = utils.train_model_stage(
    model=model_unet,
    processor=processor_unet,
    optimizer=optimizer_unet,
    loss_fn=loss_fn_unet,
    training_data_dir=TRAINING_DATA_DIR,
    stage="stage1",
    ckpt_path=ckpt_unet_s1,
    device=device,
    train_config=TRAIN_CONFIG_UNET,
    skip_training=SKIP_TRAINING_STAGE1
)

print(f"\n{'='*70}")
print("✅ STAGE 1 COMPLETE!")
print(f"{'='*70}")

## Compare Training Results (Stage 1)

Side-by-side comparison of U-Net vs LSTM training curves for Stage 1.

In [ ]:
# ============================================================================
# PLOT TRAINING COMPARISON (STAGE 1)
# ============================================================================

# Load histories from checkpoints if not available
if not hist_lstm_s1:
    ckpt_lstm = CHECKPOINT_DIR / f"model_a_lstm_stage1_{CHUNK_DURATION:.0f}s.pth"
    hist_lstm_s1 = utils.load_training_history_from_checkpoint(ckpt_lstm)

if not hist_unet_s1:
    ckpt_unet = CHECKPOINT_DIR / f"model_a_unet_stage1_{CHUNK_DURATION:.0f}s.pth"
    hist_unet_s1 = utils.load_training_history_from_checkpoint(ckpt_unet)

# Plot comparison using utility function
utils.plot_model_comparison(hist_lstm_s1, hist_unet_s1, 
                            title="Model A Comparison: LSTM vs U-Net - Stage 1")

## Stage 1 Evaluation (Spectrograms + Audio)

Compare LSTM vs U-Net separation quality on Stage 1 test samples.

In [ ]:
# ============================================================================
# STAGE 1 EVALUATION - LSTM vs U-NET (SPECTROGRAMS + AUDIO)
# ============================================================================

TRAINING_DATA_DIR = DATA_DIR / f"chunks_{CHUNK_DURATION:.0f}s"
SAMPLE_IDX = 0
DURATION = 6
SR = 22050

print("\n=== Stage 1: LSTM ===")
utils.demo_separation_sample(
    model=model_lstm,
    processor=processor_lstm,
    cache_dir=TRAINING_DATA_DIR,
    stage="stage1",
    split="test",
    song_num=SAMPLE_IDX,
    duration=DURATION,
    sr=SR,
    device=device,
    play_audio_output=True
)

print("\n=== Stage 1: U-Net ===")
utils.demo_separation_sample(
    model=model_unet,
    processor=processor_unet,
    cache_dir=TRAINING_DATA_DIR,
    stage="stage1",
    split="test",
    song_num=SAMPLE_IDX,
    duration=DURATION,
    sr=SR,
    device=device,
    play_audio_output=True
)

## Train Both Models - Stage 2

Curriculum step: Stage 2 uses 4→1 channels and continues from Stage 1 weights.

In [ ]:
# ============================================================================
# TRAIN BOTH MODEL A ARCHITECTURES - STAGE 2 (4 -> 1)
# ============================================================================

STAGE2_ENABLED = True          # Set False to skip Stage 2 entirely
SKIP_TRAINING_STAGE2 = False   # Set to True to skip Stage 2 training

print(f"\n{'='*70}")
print("STAGE 2 TRAINING: 4 → 1")
print(f"{'='*70}\n")

# Separate configs per model
TRAIN_CONFIG_LSTM = utils.get_training_config_lstm()
TRAIN_CONFIG_UNET = utils.get_training_config_unet()

# Stage 2 checkpoints
ckpt_lstm_s2 = CHECKPOINT_DIR / f"model_a_lstm_stage2_{CHUNK_DURATION:.0f}s.pth"
ckpt_unet_s2 = CHECKPOINT_DIR / f"model_a_unet_stage2_{CHUNK_DURATION:.0f}s.pth"

# Stage 1 checkpoints (for loading weights before Stage 2)
ckpt_lstm_s1 = CHECKPOINT_DIR / f"model_a_lstm_stage1_{CHUNK_DURATION:.0f}s.pth"
ckpt_unet_s1 = CHECKPOINT_DIR / f"model_a_unet_stage1_{CHUNK_DURATION:.0f}s.pth"

hist_lstm_s2 = {}
hist_unet_s2 = {}

if not STAGE2_ENABLED:
    print("⏭️  Stage 2 training disabled (STAGE2_ENABLED = False)")
else:
    print("1️⃣ Model A (LSTM) - Stage 2")
    print("-" * 70)
    model_lstm, processor_lstm, optimizer_lstm, loss_fn_lstm = utils.initialize_model_a_lstm(device)
    if ckpt_lstm_s1.exists():
        checkpoint = torch.load(ckpt_lstm_s1, map_location=device)
        model_lstm.load_state_dict(checkpoint['model_state_dict'])
        print(f"✅ Loaded Stage 1 weights: {ckpt_lstm_s1.name}")
    hist_lstm_s2 = utils.train_model_stage(
        model=model_lstm,
        processor=processor_lstm,
        optimizer=optimizer_lstm,
        loss_fn=loss_fn_lstm,
        training_data_dir=TRAINING_DATA_DIR,
        stage="stage2",
        ckpt_path=ckpt_lstm_s2,
        device=device,
        train_config=TRAIN_CONFIG_LSTM,
        skip_training=SKIP_TRAINING_STAGE2
    )

    # Free LSTM GPU memory before training U-Net
    import gc
    model_lstm = model_lstm.to('cpu')
    del optimizer_lstm
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    print("\n2️⃣ Model A (U-Net) - Stage 2")
    print("-" * 70)
    model_unet, processor_unet, optimizer_unet, loss_fn_unet = utils.initialize_model_a_unet(device)
    if ckpt_unet_s1.exists():
        checkpoint = torch.load(ckpt_unet_s1, map_location=device)
        model_unet.load_state_dict(checkpoint['model_state_dict'])
        print(f"✅ Loaded Stage 1 weights: {ckpt_unet_s1.name}")
    hist_unet_s2 = utils.train_model_stage(
        model=model_unet,
        processor=processor_unet,
        optimizer=optimizer_unet,
        loss_fn=loss_fn_unet,
        training_data_dir=TRAINING_DATA_DIR,
        stage="stage2",
        ckpt_path=ckpt_unet_s2,
        device=device,
        train_config=TRAIN_CONFIG_UNET,
        skip_training=SKIP_TRAINING_STAGE2
    )

print(f"\n{'='*70}")
print("✅ STAGE 2 COMPLETE!")
print(f"{'='*70}")

## Compare Training Results (Stage 2)

Side-by-side comparison of U-Net vs LSTM training curves for Stage 2.

In [ ]:
# ============================================================================
# PLOT TRAINING COMPARISON (STAGE 2)
# ============================================================================

if not STAGE2_ENABLED:
    print("⏭️  Stage 2 training disabled (STAGE2_ENABLED = False)")
else:
    if not hist_lstm_s2:
        ckpt_lstm_s2 = CHECKPOINT_DIR / f"model_a_lstm_stage2_{CHUNK_DURATION:.0f}s.pth"
        hist_lstm_s2 = utils.load_training_history_from_checkpoint(ckpt_lstm_s2)

    if not hist_unet_s2:
        ckpt_unet_s2 = CHECKPOINT_DIR / f"model_a_unet_stage2_{CHUNK_DURATION:.0f}s.pth"
        hist_unet_s2 = utils.load_training_history_from_checkpoint(ckpt_unet_s2)

    utils.plot_model_comparison(hist_lstm_s2, hist_unet_s2, 
                                title="Model A Comparison: LSTM vs U-Net - Stage 2")

## Stage 2 Evaluation (Spectrograms + Audio)

Compare LSTM vs U-Net separation quality on Stage 2 test samples.

In [ ]:
# ============================================================================
# STAGE 2 EVALUATION - LSTM vs U-NET (SPECTROGRAMS + AUDIO)
# ============================================================================

if not STAGE2_ENABLED:
    print("⏭️  Stage 2 evaluation skipped (STAGE2_ENABLED = False)")
else:
    SAMPLE_IDX = 0
    DURATION = 6
    SR = 22050

    print("\n=== Stage 2: LSTM ===")
    utils.demo_separation_sample(
        model=model_lstm,
        processor=processor_lstm,
        cache_dir=TRAINING_DATA_DIR,
        stage="stage2",
        split="test",
        song_num=SAMPLE_IDX,
        duration=DURATION,
        sr=SR,
        device=device,
        play_audio_output=True
    )

    print("\n=== Stage 2: U-Net ===")
    utils.demo_separation_sample(
        model=model_unet,
        processor=processor_unet,
        cache_dir=TRAINING_DATA_DIR,
        stage="stage2",
        split="test",
        song_num=SAMPLE_IDX,
        duration=DURATION,
        sr=SR,
        device=device,
        play_audio_output=True
    )

## Quantitative Evaluation

Compute BSS metrics (SDR/SIR/SAR) on test set using museval library.

In [ ]:
# ============================================================================
# QUANTITATIVE EVALUATION - BSS METRICS
# ============================================================================

NUM_TEST_SAMPLES = 10
STAGE_FOR_EVAL = "stage2" if STAGE2_ENABLED else "stage1"

# Evaluate both models using utility function
metrics = utils.evaluate_separation_quality(
    model_lstm=model_lstm,
    model_unet=model_unet,
    processor_lstm=processor_lstm,
    processor_unet=processor_unet,
    test_data_dir=TRAINING_DATA_DIR,
    stage=STAGE_FOR_EVAL,
    num_samples=NUM_TEST_SAMPLES,
    sr=22050,
    device=device
)

## Optional: Custom Song Inference (Upload)

Upload a song (or place it in the folder below) and run inference with both models. This is the final step.

In [ ]:
# ============================================================================
# CUSTOM SONG INFERENCE (UPLOAD OR LOCAL FOLDER)
# ============================================================================

UPLOAD_DIR = DATA_DIR / "user_uploads"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Colab upload
if IN_COLAB:
    try:
        from google.colab import files
        uploaded = files.upload()
        for name, data in uploaded.items():
            out_path = UPLOAD_DIR / name
            with open(out_path, "wb") as f:
                f.write(data)
            print(f"✅ Saved: {out_path}")
    except Exception as e:
        print(f"⚠️  Colab upload failed: {e}")
        print(f"Place a file manually in: {UPLOAD_DIR}")
else:
    print(f"📁 Place your audio file in: {UPLOAD_DIR}")

# Pick the most recent audio file
exts = ["*.wav", "*.mp3", "*.flac", "*.ogg", "*.m4a"]
audio_files = []
for ext in exts:
    audio_files += list(UPLOAD_DIR.glob(ext))

audio_files = sorted(audio_files, key=lambda p: p.stat().st_mtime, reverse=True)

if not audio_files:
    print("⚠️  No audio files found in upload folder.")
else:
    audio_path = audio_files[0]
    print(f"🎵 Using file: {audio_path.name}")

    utils.compare_models_on_audio_file(
        file_path=audio_path,
        model_lstm=model_lstm,
        model_unet=model_unet,
        processor_lstm=processor_lstm,
        processor_unet=processor_unet,
        device=device,
        sr=22050,
        duration=15  # seconds (set None for full length)
    )